# Intro

This notebook shows how to run the corruption pipeline end-to-end: inject known data-quality issues into the clean OCEL SQLite log, then sweep every detector over the result.

Corruption logic lives in `src/corruption.py`. The single entry point is `corrupt_database(src, dst, level=...)` — see the next cell.

## Run the corruption

`level` accepts `'legacy'`, `'easy'`, `'medium'`, `'hard'`, or `'all'`. `'all'` runs every injector (24 total across 8 issue types); each tiered level runs one flavor per issue. When `dst_path=None`, the output path defaults to `data/order-management-<level>.sqlite` (or `-full.sqlite` for `'all'`).

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.corruption import corrupt_database, DEFAULT_CLEAN_PATH

DB = corrupt_database(DEFAULT_CLEAN_PATH, level="all")
print(f"Corrupted database: {DB}")

Corrupted database: /Users/I762870/dev-priv/ocel-healer/data/order-management-full.sqlite


## Full detector sweep

`detect_all` runs every rule-based detector and returns one Polars DataFrame per issue. Use this rather than reimplementing per-issue queries in the notebook.

In [2]:
from src.detection.error_detection import detect_all

results = detect_all(DB)
print("Detector counts:")
for issue, df in results.items():
    print(f"  {issue:34s} {df.height}")

# Peek at each detector's output — one row per detected violation.
for issue, df in results.items():
    if df.height == 0:
        continue
    print(f"\n=== {issue} ({df.height} rows) ===")
    print(df.head(5))

Detector counts:
  missing_object_type                4
  missing_attribute_value            3
  duplicate_objects_on_ids           3
  duplicate_objects_on_attributes    387
  incorrect_attribute_datatype       3
  dangling_o2o_relationship          7
  dangling_e2o_relationship          464

=== missing_object_type (4 rows) ===
shape: (4, 3)
┌──────────────────────────┬───────────┬─────────────────────┐
│ ocel_id                  ┆ ocel_type ┆ issue               │
│ ---                      ┆ ---       ┆ ---                 │
│ str                      ┆ str       ┆ str                 │
╞══════════════════════════╪═══════════╪═════════════════════╡
│ Wil van der Aalst        ┆ null      ┆ missing_object_type │
│ MacBook Pro              ┆           ┆ missing_object_type │
│ o-990010                 ┆           ┆ missing_object_type │
│ AlpenTech Innovations AG ┆ null      ┆ missing_object_type │
└──────────────────────────┴───────────┴─────────────────────┘

=== missing_attribute_v